# Chatbot FAQ

In [14]:
import sys
from pathlib import Path

import pandas as pd

In [15]:
project_root = Path.cwd()
candidate_dir = project_root / "Semana_1_Proyecto_4"
project_dir = candidate_dir if candidate_dir.exists() else project_root

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from chatbot_utils import DEFAULT_THRESHOLD, load_faq, predict_response, save_artifacts, train_artifacts

FAQ_PATH = project_dir / "chatbot_faq.json"
ARTIFACT_PATH = project_dir / "chatbot_tfidf_artifacts.joblib"
METADATA_PATH = project_dir / "chatbot_tfidf_metadata.json"
CONFIDENCE_THRESHOLD = DEFAULT_THRESHOLD

faq_items = load_faq(FAQ_PATH)
print(f"FAQs cargadas: {len(faq_items)} | Threshold: {CONFIDENCE_THRESHOLD:.2f}")

faq_preview = pd.DataFrame(
    [
        {"id": item["id"], "categoria": item.get("category", "general"), "pregunta": item["questions"][0]}
        for item in faq_items
    ]
)
faq_preview

FAQs cargadas: 6 | Threshold: 0.30


,id,categoria,pregunta
0,horario_atencion,atencion,Cual es el horario de atencion
1,ubicacion_tienda,ubicacion,Donde esta ubicada la tienda
2,metodos_pago,pagos,Cuales son los metodos de pago disponibles
3,envios_nacionales,envios,Tienen envios a todo el pais
4,productos_stock,catalogo,Que productos tienen en stock
5,devoluciones,postventa,Como puedo hacer una devolucion


In [16]:
artifacts = train_artifacts(FAQ_PATH, threshold=CONFIDENCE_THRESHOLD)
print(f"Preguntas indexadas: {len(artifacts['answer_lookup'])}, Vocabulario: {len(artifacts['vectorizer'].vocabulary_)}")

Preguntas indexadas: 30, Vocabulario: 87


In [17]:
sample_queries = [
    "A que hora atienden",
    "Donde queda la tienda",
    "Aceptan transferencia bancaria",
    "Quiero soporte para un celular",
    "Necesito hablar con alguien",
]

results = [predict_response(query, artifacts) for query in sample_queries]

evaluation_df = pd.DataFrame(
    [
        {
            "consulta": query,
            "estado": result["status"],
            "score": round(result["score"], 3),
            "respuesta": result["answer"],
            "sugerencias": ", ".join(result["suggestions"]),
        }
        for query, result in zip(sample_queries, results)
    ]
)

save_artifacts(ARTIFACT_PATH, METADATA_PATH, artifacts, faq_items)
print(f"Artefacto: {ARTIFACT_PATH}", f"Metadata: {METADATA_PATH}", sep="\n")

evaluation_df

Artefacto: c:\GitHub\Analitica-Negocios\Semana_1_Proyecto_4\chatbot_tfidf_artifacts.joblib
Metadata: c:\GitHub\Analitica-Negocios\Semana_1_Proyecto_4\chatbot_tfidf_metadata.json


,consulta,estado,score,respuesta,sugerencias
0,A que hora atienden,match,1.000,Nuestro horario de atencion es de lunes a vier...,
1,Donde queda la tienda,match,0.499,Nuestra tienda esta ubicada en Calle Ficticia ...,
2,Aceptan transferencia bancaria,match,0.408,"Aceptamos pagos con tarjeta de credito, debito...",
3,Quiero soporte para un celular,fallback,0.000,No estoy totalmente seguro de tu consulta. Pue...,"Como puedo hacer una devolucion, Que productos..."
4,Necesito hablar con alguien,fallback,0.000,No estoy totalmente seguro de tu consulta. Pue...,"Como puedo hacer una devolucion, Que productos..."
